> **You're profiling a 7B-parameter language model serving 200 requests per second. GPU utilization: 97%. Latency at S=1024: 85 ms — well above the 40 ms SLA. You open the profiler. One line dominates: `softmax(Q @ K.T / sqrt(d))` at 61% of attention wall time. The tensor cores are idle 70% of the time. Compute isn't the bottleneck.**
>
> **You try a fused softmax kernel — no change. You switch to fp16 — no improvement. Then you look at the memory-bandwidth monitor: it's pinned at 98% utilization. The attention layer isn't slow because of arithmetic — it's slow because of data movement. The S×S attention score matrix is written to and read back from HBM (high-bandwidth memory) five times per forward-backward pass. At S=1024, that's 400 MB of HBM traffic per layer, per step. Double the sequence length? The traffic quadruples.**
>
> **There exists an algorithm that runs the exact same attention math — no approximation, no quality loss — while eliminating those HBM roundtrips entirely. It keeps the score matrix in on-chip SRAM. It computes attention in tiles. It's called FlashAttention, and it's already inside `F.scaled_dot_product_attention`. This notebook shows you exactly how it works — from the memory accounting to the tiling loop to the one dispatch condition that determines whether you're actually getting it.**

# FlashAttention: Inside the Algorithm| Part | Concept | Why you need this right now ||------|---------|---------------------------|| 1 | Standard attention's HBM problem | You can't fix what you can't measure — quantify exactly how much HBM traffic the S×S matrix generates before proposing a solution || 2 | Tiling insight | HBM is slow (2 TB/s); on-chip SRAM is 9× faster but tiny — tiling bridges them without any approximation || 3 | Online softmax | Tiling breaks the row-dependency in softmax; you must prove the incremental version is algebraically exact before trusting it in production || 4 | IO complexity | Intuition says "faster" — the arithmetic tells you *how much faster* and under what conditions the win disappears || 5 | PyTorch 2.0 dispatch | A one-line change gets you the speedup for free — but only when specific dtype/shape conditions are met || 6 | GQA and MQA | FlashAttention fixes attention compute; the KV cache is the next bottleneck at inference time — GQA solves it |---## Prerequisite Bridge — From Ch1 and Ch3| Foundation | Role in this notebook ||---|---|| HBM bandwidth (~2 TB/s on A100) | IO complexity matters because every HBM read/write costs ~0.5 ns || Arithmetic intensity | Standard attention's S×S softmax is memory-bound (low FLOP/byte) || Memory-bound bottleneck identified | Ch3 showed softmax takes longer than matmul — this chapter fixes it |

## Working Vocabulary| Term | Meaning in this notebook ||---|---|| **HBM** | High-bandwidth memory: large off-chip GPU memory. Tensors live here, but each trip to a compute unit is costly. || **SRAM** | Small, fast on-chip memory. It is roughly an order of magnitude faster than HBM but cannot hold a full attention matrix. || **Tile** | A rectangular tensor block sized to fit on chip, such as some Q rows crossed with some K rows. || **Kernel** | One GPU program launch. A fused kernel completes several math steps before writing intermediates to HBM. || **Materialization** | Creating and storing a complete intermediate tensor. Eager attention materializes S×S scores and probabilities. || **Online softmax** | A reformulation that updates a row's running maximum, normalization sum, and weighted-value accumulator as tiles arrive. || **Causal mask** | The rule that token i cannot attend to future token j > i; fully masked future tiles can be skipped. || **GQA** | Grouped Query Attention: groups of query heads share K/V heads, shrinking the KV cache. || **MQA** | Multi-Query Attention: all query heads share one K head and one V head, the maximum-sharing endpoint of GQA. |**Mental model:** HBM is the warehouse, SRAM is the workbench, a tile is one tray, and the kernel finishes as much as possible before returning results to the warehouse.

In [ ]:
import subprocess, sys

# Install any of these three packages that aren't already available
for pkg in ['torch', 'numpy', 'matplotlib']:
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time

# Use CUDA if a GPU is visible to PyTorch, otherwise fall back to CPU
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
HAS_GPU = torch.cuda.is_available()
torch.manual_seed(42)

print(f"Device: {DEVICE}")

# Pick a reference HBM bandwidth for the IO-traffic estimates below
if HAS_GPU:
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}")
    HBM_BANDWIDTH_TBS = 2.0  # A100 reference; adjust for actual GPU
else:
    print("No GPU — all benchmarks run on CPU. IO measurements show relative ratios.")
    HBM_BANDWIDTH_TBS = 0.05  # CPU memory bandwidth (approximate)

print()

#  Running example
B, D = 8, 64  # batch=8, head_dim=64 (one attention head)
print(f"Running example: single attention head (B={B}, D={D})")
print(f"Sequence length scales: S=128 → 512 → 2048 across this notebook")


---

## Part 1 — Standard Attention: The S×S Memory Problem

Standard attention computes:

$$\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d}}\right)V$$

This requires materializing the full $(S \times S)$ attention score matrix **three separate times** during the forward pass:
1. After `matmul(Q, K.T)` → write scores to HBM
2. During `softmax` → read scores, write softmax output to HBM
3. During `matmul(softmax_out, V)` → read softmax output

During the backward pass: each of Q, K, V requires reading the attention matrix again → **3 more HBM roundtrips**.

Total HBM traffic for standard attention: proportional to **6 × S × S × d_head** bytes.

#### #### Predict first

Standard attention reads the S×S matrix N times. Tiled FlashAttention avoids materializing it. Which statement is correct?

1. **(a) Standard attention requires ~5× more HBM reads than FlashAttention** — the S×S matrix is read repeatedly for backward
2. **(b) Standard reads ~3× more** — one for forward softmax, two for backward
3. **(c) They're approximately equal** — FlashAttention recomputes during backward, using more FLOPs

### Anchor 1 — Follow the avoidable HBM crossingsBefore looking at the image, trace one score: Q and K are read, the score is written, softmax reads it, a probability is written, and the final matmul reads it. The multiply is not the surprise; letting the S×S intermediates leave the chip is.![Standard attention HBM roundtrips: Q/K/V read, S=QKᵀ written to HBM, softmax written, output PV written — O(S²) total](images/standard-attention-io.png)**Guided reading:** count arrows crossing the HBM boundary. Q/K/V and the final output are necessary traffic. The full score and probability matrices are avoidable traffic between separate kernels.```mermaidflowchart LR H1[(HBM: Q and K)] -->|read| A[QKᵀ kernel] A -->|write S×S| H2[(HBM: scores)] H2 -->|read| B[softmax kernel] B -->|write S×S| H3[(HBM: probabilities)] H3 -->|read| C[PV kernel] H4[(HBM: V)] -->|read| C C -->|write once| H5[(HBM: output)]```Fast kernels can still form a slow pipeline when quadratic intermediates are materialized between them.

In [ ]:
#  Part 1: Measure standard attention memory traffic
def standard_attention(Q, K, V, scale):
    """Standard attention — materializes full S×S score matrix."""
    scores = torch.matmul(Q, K.transpose(-2, -1)) * scale  # (B, S, S)
    attn   = torch.softmax(scores, dim=-1)                 # (B, S, S)
    return torch.matmul(attn, V)                            # (B, S, D)

def measure_hbm_traffic_gb(B, S, D, dtype=torch.float32):
    """Estimate HBM traffic for standard attention (forward only)."""
    bytes_per_elem = 4 if dtype == torch.float32 else 2

    # Q, K, V reads: 3 × B × S × D
    qkv_read = 3 * B * S * D * bytes_per_elem

    # Score matrix write + read (matmul output → softmax input): 2 × B × S × S
    score_write_read = 2 * B * S * S * bytes_per_elem

    # Softmax output write + matmul read: 2 × B × S × S
    attn_write_read = 2 * B * S * S * bytes_per_elem

    # Output write: B × S × D
    output_write = B * S * D * bytes_per_elem
    total = qkv_read + score_write_read + attn_write_read + output_write
    return total / 1e9  # GB

seq_lengths = [128, 256, 512, 1024, 2048]
print("Standard attention HBM traffic (forward pass only):")
print(f"{'S':6s}  {'HBM GB':8s}  {'S×S matrix':10s}  {'Compute ms (ref)':16s}")
print("-" * 50)

# Show how HBM traffic grows with sequence length for standard (non-tiled) attention
for S_val in seq_lengths:
    hbm_gb = measure_hbm_traffic_gb(B, S_val, D)
    score_gb = 2 * B * S_val * S_val * 4 / 1e9  # score write+read
    est_time_ms = hbm_gb / (HBM_BANDWIDTH_TBS * 1000) * 1000  # ms
    print(f"  {S_val:4d}   {hbm_gb:6.3f} GB   {score_gb:6.3f} GB     ~{est_time_ms:.2f}ms")

print()
print("Key insight: HBM traffic grows as O(S²) due to the score matrix.")
print("At S=2048: S×S = 4M elements × 4 bytes × 2 (write+read) × B = {:.1f} GB".format(
    2 * B * 2048 * 2048 * 4 / 1e9))
print()
print("Prediction check: answer (a) — accounting for backward pass (3 more reads), total is ~5-6× more")


In [ ]:
#  Part 1: IO traffic vs sequence length
S_range = np.arange(64, 2049, 64)

# Compute standard attention's HBM traffic across the whole sequence-length sweep
hbm_fwd = [measure_hbm_traffic_gb(B, int(s), D) for s in S_range]

# FlashAttention: O(S²/M) where M is SRAM size; roughly O(S) for practical block sizes
# Reference: FlashAttention-2 paper Figure 2: ~1/5 the HBM traffic of standard
hbm_flash = [measure_hbm_traffic_gb(B, int(s), D) / 5 for s in S_range]  # approximate

# Plot standard (forward and fwd+bwd) vs FlashAttention HBM traffic, with the savings shaded in
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(S_range, hbm_fwd,   'coral',       lw=2, label='Standard attention (forward)')
ax.plot(S_range, [h*2 for h in hbm_fwd], 'coral', lw=2, ls='--', label='Standard (fwd+bwd ≈ 6× reads)')
ax.plot(S_range, hbm_flash, 'steelblue',   lw=2, label='FlashAttention (approx.)')
ax.fill_between(S_range, hbm_flash, [h*2 for h in hbm_fwd], alpha=0.1, color='mediumseagreen',
                label='HBM savings from FlashAttention')
ax.set_xlabel('Sequence length S'); ax.set_ylabel('HBM traffic (GB)')
ax.set_title('HBM traffic: Standard attention vs. FlashAttention'); ax.legend()
ax.axvline(512, color='gray', ls=':', lw=1, label='S=512 (Ch3 bottleneck)')
plt.tight_layout(); plt.show()
print("→ The gap between standard and FlashAttention grows quadratically with sequence length.")


#### What just happened — and what's missing

The O(S²) cost is now visceral: at S=2048 with B=8, standard attention generates ~12 GB of HBM traffic for a full forward+backward pass. The score matrix — 4 bytes × S² elements — gets written once (after matmul), read once (for softmax), written again (softmax output), read again (final matmul) — then all three reads repeat for each backward pass gradient.

**What's missing:** We know the S×S matrix is the bottleneck. But we *need* it — every output token attends over all input tokens. How do we compute attention without materializing that matrix in HBM? Part 2 shows the tiling trick that makes it possible.

---

## Part 2 — Tiling Insight: Keep the S×S Matrix in SRAM

**Key idea:** The S×S attention matrix is too large for SRAM (228 KB/SM on A100), but we can process it in **tiles** (blocks) that fit in SRAM.

Algorithm outline (FlashAttention Algorithm 1):
1. Load a tile of Q: `Q_tile = Q[i:i+BLOCK, :]`
2. For each tile of K, V: `K_tile = K[j:j+BLOCK, :]`
3. Compute tile of attention scores: `S_ij = Q_tile @ K_tile.T`
4. Accumulate output with online softmax correction (Part 3)
5. Write output tile back to HBM — only once

The S×S matrix is **never written to HBM** — only SRAM is used for intermediate computation.

> **Deriving tiling — don't just accept it:** At S=512 with B=8, the score matrix S_ij = Q @ Kᵀ has shape (8, 512, 512). At 4 bytes per float: 8 × 512 × 512 × 4 = **8 MB**. On-chip SRAM on A100 is 228 KB per streaming multiprocessor. You cannot hold the 8 MB matrix in SRAM. But here's the key: you don't need all of S_ij at once. Each row of the output only depends on the corresponding row of Q and the entire K/V matrices. What is the smallest unit of computation that is self-contained? A block of Q_i rows against a block of K_j rows. Compute one (Q_i × K_j) tile, accumulate into the output tile O_i, move to the next j — never materializing the full 8 MB matrix. That's tiling.

### Anchor 2 — Watch a score tile be born and die on chipBefore looking at the image, ask what must survive after one K/V tile is consumed. The score tile does not: only running `(m, l, O)` state for the current Q tile survives.![FlashAttention tiling: Q/K/V loaded in tiles to SRAM; the full S×S matrix is never written to HBM — O(S²/M) reads](images/flash-attention-tiling.png)**Guided reading:** a Q tile stays on the workbench while K/V tiles stream past it. Each score tile contributes to `(m, l, O)` and is discarded. The completed output tile is written to HBM once.```mermaidflowchart LR Q[Load Q tile] --> KV[Load next K/V tile] KV --> S[Compute score tile in SRAM] S --> M[Apply causal mask if needed] M --> U[Update and rescale m, l, O] U --> D{More K/V tiles?} D -->|yes| KV D -->|no| W[Normalize and write output tile once]```This is the tile lifecycle: load, compute, merge, discard, repeat.

#### #### Predict first

The tiled attention algorithm claims it produces the **exact same output** as standard attention — despite never writing the full S×S matrix to HBM. Before running the code, which is true?

1. **(a) The output is approximate** — tiling introduces rounding error (~1e-4 absolute difference from the reference)
2. **(b) The output is numerically identical** — tiling is a mathematically exact decomposition, not an approximation
3. **(c) Identical only when S is a multiple of BLOCK_SIZE** — otherwise boundary tiles introduce error

*Think it through: does the mathematics of softmax allow partial summation? Look at how the running max/sum correction terms are derived in the tiling loop.*

In [ ]:
#  Part 2: Tiling algorithm walkthrough
def tiled_attention_forward(Q, K, V, BLOCK_SIZE=32):
    """
    Tiled attention following Algorithm 1 of FlashAttention paper.

    This is a readable Python implementation — equivalent to the paper's pseudocode
    but written for clarity, not speed. The Triton/CUDA implementation (Ch8) achieves
    the actual HBM savings.

    Key property: the full (S×S) score matrix is never materialized.
    Only (BLOCK × BLOCK) tiles are computed at a time.
    """
    B_size, S, D_head = Q.shape
    scale = D_head ** -0.5

    # Output accumulator
    O = torch.zeros_like(Q)              # (B, S, D)
    L = torch.zeros(B_size, S)           # normalisation factor (sum of softmax weights)
    M = torch.full((B_size, S), -float('inf'))  # running max for numerical stability

    # Tile over sequence dimension
    for i in range(0, S, BLOCK_SIZE):
        Q_i = Q[:, i:i+BLOCK_SIZE, :]           # (B, BLOCK, D) — in SRAM
        M_i = M[:, i:i+BLOCK_SIZE].clone()      # running max for this tile
        L_i = L[:, i:i+BLOCK_SIZE].clone()      # running sum for this tile
        O_i = O[:, i:i+BLOCK_SIZE, :].clone()   # output accumulator for this tile

        # Stream K/V tiles one at a time, updating the running softmax stats online
        for j in range(0, S, BLOCK_SIZE):
            K_j = K[:, j:j+BLOCK_SIZE, :]       # (B, BLOCK, D) — in SRAM
            V_j = V[:, j:j+BLOCK_SIZE, :]       # (B, BLOCK, D) — in SRAM

            # Score tile: (B, BLOCK_i, BLOCK_j) — computed and immediately used
            S_ij = torch.matmul(Q_i, K_j.transpose(-2, -1)) * scale

            # new running max across all j-tiles seen so far for this Q_i block
            M_new = torch.maximum(M_i, S_ij.max(dim=-1).values)

            # numerically stable: shift by new max so exp() doesn't overflow
            exp_S  = torch.exp(S_ij - M_new.unsqueeze(-1))

            # correct old sum: rescale for the new (higher) max, then add this tile's mass
            L_new  = torch.exp(M_i - M_new) * L_i + exp_S.sum(dim=-1)

            # term 1: rescale the old accumulated output to the new max scale
            # term 2: add this tile's weighted V contribution
            O_new  = (torch.exp(M_i - M_new).unsqueeze(-1) * O_i +
                      torch.matmul(exp_S, V_j))

            M_i, L_i, O_i = M_new, L_new, O_new

        # Write output tile back to HBM — only once per Q tile
        O[:, i:i+BLOCK_SIZE, :] = O_i / L_i.unsqueeze(-1)  # normalise

    return O

# Test on a small example
S_test = 64
Q_t = torch.randn(2, S_test, D); K_t = torch.randn(2, S_test, D); V_t = torch.randn(2, S_test, D)
scale = D ** -0.5

tiled_out = tiled_attention_forward(Q_t, K_t, V_t, BLOCK_SIZE=16)
ref_out   = F.scaled_dot_product_attention(Q_t, K_t, V_t, scale=scale)

match = torch.allclose(tiled_out, ref_out, atol=1e-5)
print(f"Tiled attention output matches reference: {match}")
print(f"  Max absolute error: {(tiled_out - ref_out).abs().max():.2e}")
print()
print("The S×S matrix was NEVER materialized in HBM during tiled computation.")
print(f"  Largest intermediate tensor: ({2}×{16}×{16}) = {2*16*16*4/1024:.1f} KB (fits in SRAM)")
print(f"  vs standard S×S: ({2}×{S_test}×{S_test}) = {2*S_test*S_test*4/1024:.1f} KB")


#### What just happened — and what's missing

The tiled algorithm produces output indistinguishable from standard attention (`torch.allclose`, atol=1e-5). Prediction answer: **(b)** — the tiling is mathematically exact, not an approximation.

The S×S matrix was never written to HBM. The largest intermediate tensor was a `(B × BLOCK × BLOCK)` score tile — a few KB that fits in SRAM rather than the multi-MB full matrix.

**What's missing:** The tiling loop uses a *running max* and *running sum* to correct the output as each tile arrives. We called this "online softmax" and assumed it is exact. Is it really numerically identical to `torch.softmax`? Part 3 proves this rigorously — including under extreme inputs that would normally overflow exp().

####  Your Turn — Tiling Block Size

Does the tiled algorithm's correctness depend on the block size? Standard softmax reads the full row — our tiled version processes it in chunks with an online correction. The claim: any BLOCK_SIZE that divides S evenly gives numerically identical results.

Change `BLOCK_SIZE_TRY` and verify whether the algorithm is always exact — or whether some block sizes break it.

In [ ]:
#   Your Turn: Block size effect on numerical exactness
# # CHANGE: try different block sizes and observe whether correctness holds
BLOCK_SIZE_TRY = 16  # ← CHANGE ME: try 8, 32, 64 (any value that divides S_test=64)

torch.manual_seed(42)
S_try = 64
Q_try = torch.randn(2, S_try, D)
K_try = torch.randn(2, S_try, D)
V_try = torch.randn(2, S_try, D)

out_tiled_try = tiled_attention_forward(Q_try, K_try, V_try, BLOCK_SIZE=BLOCK_SIZE_TRY)
out_ref_try   = F.scaled_dot_product_attention(Q_try, K_try, V_try, scale=D**-0.5)

err_try = (out_tiled_try - out_ref_try).abs().max().item()
match_try = err_try < 1e-5
print(f"Block size {BLOCK_SIZE_TRY:3d}: max absolute error = {err_try:.2e}  —  match: {' exact' if match_try else ' above tolerance'}")
print()

# Report whether the tiled result matched the reference within tolerance
if match_try:
    print("→ Exact for this block size. Try 8, 32, 64 — the answer should always be ' exact'.")
    print("→ The online softmax correction is algebraically exact regardless of block size.")
else:
    print("→ Check whether BLOCK_SIZE_TRY divides S_try=64 evenly. Boundary mismatch causes error.")


---## Part 3 — Online Softmax: Stable Computation Without the Full Row**The challenge:** Standard softmax needs all $S$ values at once to compute the denominator $\sum_j e^{x_j}$. If we're processing tiles, we only see a block at a time.**Online softmax solution (Milakov & Gimelshein, 2018):** maintain running statistics $(m, \ell)$ — the running maximum and sum. As each tile arrives, update these statistics and correct the running output.The correction preserves the **same mathematical softmax**; compare floating-point implementations within tolerance.

> **Intuition:** Think of computing the class average at a company you're visiting one department at a time. When you reach a department with exceptionally high salaries (a new maximum), you correct your running average using a mathematical factor rather than re-visiting all previous departments. The correction factor `exp(old_max - new_max)` does exactly that for our softmax — it rescales the old accumulated output to account for the new, higher maximum, so the final result is mathematically identical to computing softmax over the entire row at once.

#### #### Predict firstOnline softmax maintains a running max `m` and running sum `l`. When a new tile arrives with a higher max, it applies a correction `exp(old_max - new_max)` to the accumulated sum. Will this produce:1. **(a) The same mathematical output as `torch.softmax`, within floating-point tolerance** — the correction is algebraically exact2. **(b) Very close but not identical** — floating-point rounding in the correction term introduces ~1e-7 error3. **(c) Unstable for extreme inputs** — the `exp()` in the correction overflows when one value is much larger than the others*Hint: think about what the correction `exp(old_max - new_max)` guarantees. The exponent is always ≤ 0 — why does that matter?*

In [ ]:
#  Part 3: Online softmax implementation and verification
def online_softmax(x_seq):
    """
    Compute softmax of x_seq using the online (one-pass) algorithm.
    Processes x_seq in BLOCK_SIZE chunks, maintaining running max (m) and sum (l).
    Produces bit-identical results to torch.softmax.
    """
    S_len = x_seq.shape[-1]
    BLOCK = 16

    m = torch.full(x_seq.shape[:-1], -float('inf'))  # running max
    l = torch.zeros_like(m)                             # running sum
    result = torch.zeros_like(x_seq)

    # Process the sequence in fixed-size blocks, updating the running max/sum online
    for j in range(0, S_len, BLOCK):
        x_block = x_seq[..., j:j+BLOCK]
        m_block  = x_block.max(dim=-1).values

        # Update running max
        m_new = torch.maximum(m, m_block)

        # Correct existing result for new max
        l_new = torch.exp(m - m_new) * l + torch.exp(x_block - m_new.unsqueeze(-1)).sum(dim=-1)
        result[..., :j+BLOCK] = torch.exp(x_seq[..., :j+BLOCK] - m_new.unsqueeze(-1)) / l_new.unsqueeze(-1)

        m, l = m_new, l_new

    return result

# Verify against torch.softmax
torch.manual_seed(42)
x_test = torch.randn(4, 128) * 3  # amplified to stress numerical stability

out_online  = online_softmax(x_test)
out_pytorch = torch.softmax(x_test, dim=-1)

match = torch.allclose(out_online, out_pytorch, atol=1e-5)
max_err = (out_online - out_pytorch).abs().max().item()
print(f"Online softmax matches torch.softmax: {match}")
print(f"  Max absolute error: {max_err:.2e}  (atol=1e-5: {'' if max_err < 1e-5 else ''})")
print()
print("Key property: online softmax produces identical results to standard softmax")
print("  but processes data in tiles, never requiring the full row in memory simultaneously.")
print()

# Demonstrate numerical stability with a challenging input
x_extreme = torch.tensor([[100.0, 101.0, 50.0, -100.0]])  # large values
out_stable = online_softmax(x_extreme)
out_ref    = torch.softmax(x_extreme, dim=-1)
print(f"Numerical stability test (extreme values: 100, 101, 50, -100):")
print(f"  Online:    {out_stable.numpy()}")
print(f"  Reference: {out_ref.numpy()}")
print(f"  Match: {torch.allclose(out_stable, out_ref, atol=1e-5)}")


### Why online softmax is exactFor a score row split into blocks, standard softmax finds global $m$ and $\ell=\sum_j e^{x_j-m}$. The tiled form merges each block into the same statistics:$$m_{new}=\max(m_{old},m_{tile})$$$$\ell_{new}=e^{m_{old}-m_{new}}\ell_{old}+\sum_{j\in tile}e^{x_j-m_{new}}$$```mermaidflowchart TB X[Same complete score row] --> A[Standard: global max and sum] X --> B[Tiled: merge block max and sums] A --> C[Same mathematical m and l] B --> C C --> O[Same attention output]```The exponential factor converts old partial state to the new maximum's scale before addition; no score is dropped. This is **algebraic exactness**, not approximation. Floating-point reduction order can change the last bits, so `torch.allclose` is the correct implementation check rather than byte-for-byte equality.

#### What just happened — and what's missingOnline softmax produces the same mathematical output as standard softmax within floating-point tolerance but processes data in tiles. Combined with the tiling algorithm from Part 2, this means the entire forward pass can be computed without ever writing the S×S attention matrix to HBM.**Missing piece:** We've proved the algorithm is correct and avoids HBM writes. But how much faster is it actually? The theoretical IO complexity predicts large savings — but does the hardware agree? That's Part 4.

####  Your Turn — Numerical Stability Under Extreme Inputs

Online softmax claims to be numerically stable even for extreme values. The correction `exp(old_max - new_max)` ensures exp() never overflows because the exponent is always ≤ 0. Stress-test this guarantee by scaling the inputs to increasingly large magnitudes.

In [ ]:
#   Your Turn: Stress-test online softmax numerical stability
# # CHANGE: modify extreme_scale to see at what input magnitude online softmax breaks (if ever)
extreme_scale = 50.0  # ← CHANGE ME: try 10, 100, 500, 1000

torch.manual_seed(7)
x_stress = torch.randn(3, 256) * extreme_scale

out_online_stress  = online_softmax(x_stress)
out_pytorch_stress = torch.softmax(x_stress, dim=-1)

err_stress = (out_online_stress - out_pytorch_stress).abs().max().item()
print(f"Scale = {extreme_scale}: max absolute error = {err_stress:.2e}")
print()

# Flag whether the online algorithm held up numerically at this magnitude
if err_stress < 1e-5:
    print("→ Online softmax remains stable. The running-max trick absorbs the large values.")
    print("  exp(old_max - new_max) ≤ exp(0) = 1.0, so the correction never overflows.")
else:
    print("→ Error above tolerance. This may indicate a boundary condition — check BLOCK vs. S.")
print()
print("Key insight: the running-max normalization is the secret weapon against overflow.")
print("Without it, large raw scores would push exp() to inf before normalization could occur.")


---

## Part 4 — IO Complexity: Measuring the HBM Traffic Reduction

Standard attention requires O(S²) HBM reads (the score matrix is read multiple times during the backward pass). FlashAttention's tiling achieves O(S²/M) reads. Let's measure the actual timing difference on our running example.

> **Making the cost visceral:** At 2 TB/s HBM bandwidth, reading 1 GB takes 0.5 ms. Standard attention at S=512 generates roughly 0.05 GB of S×S traffic for the forward pass alone — and six times that for a full forward+backward. That's 0.15 ms just waiting for the weight bus, on a GPU whose tensor cores could have done the actual computation in 0.02 ms. The profiler from Ch3 measured exactly this gap: softmax was slower than matmul despite fewer FLOPs because the S×S matrix requires repeated HBM roundtrips. The O(S²/M) formula is the accounting for that bus-time.

#### #### Predict first

FlashAttention reduces HBM traffic from O(S²) to roughly O(S). At S=512 with B=4 on CPU, what speedup over standard attention does `F.scaled_dot_product_attention` actually achieve?

1. **(a) <1.5×** — on CPU, FlashAttention's HBM-bandwidth benefit doesn't apply; tiling overhead negates any gain
2. **(b) 2–4×** — even on CPU, fused kernel avoids Python dispatch overhead and materializing the intermediate score matrix
3. **(c) 5–10×** — the O(S²) vs O(S) memory traffic difference dominates even on CPU at this sequence length

*Note: on an A100 GPU at S=2048, the expected speedup is 3–5× — driven by HBM bandwidth reduction, not compute.*

In [ ]:
#  Part 4: IO complexity analysis
print("IO Complexity Analysis:")
print()
print("Standard attention (forward):")
print("  Reads:  Q + K + V + S + attn = (3×B×S×D + 2×B×S×S) floats")
print("  Writes: S + attn + O           = (2×B×S×S + B×S×D) floats")
print("  Total:  O(S²) — dominated by the S×S terms")
print()
print("FlashAttention (forward):")
print("  Reads:  Q + K + V = 3×B×S×D floats  (no S×S reads!)")
print("  Writes: O = B×S×D floats             (one output write)")
print("  Total:  O(S) — linear, not quadratic")
print()

# Measure timing ratio across sequence lengths
B_bench = 4
D_bench = 64
seq_lengths = [128, 256, 512]

print(f"Timing comparison (B={B_bench}, D={D_bench}):")
print(f"{'S':6s}  {'Standard (ms)':14s}  {'torch.SDPA (ms)':16s}  {'Speedup':8s}")
print("-" * 50)

# Compare hand-written standard attention against PyTorch's dispatch-optimized SDPA
for S_val in seq_lengths:
    Q_b = torch.randn(B_bench, S_val, D_bench).to(DEVICE)
    K_b = torch.randn(B_bench, S_val, D_bench).to(DEVICE)
    V_b = torch.randn(B_bench, S_val, D_bench).to(DEVICE)

    # Time fn() n times with GPU sync around each call, return median ms
    def bench(fn, n=30):
        if HAS_GPU: torch.cuda.synchronize()
        times = []
        for _ in range(n):
            if HAS_GPU: torch.cuda.synchronize()
            t0 = time.perf_counter()
            fn()
            if HAS_GPU: torch.cuda.synchronize()
            times.append(time.perf_counter() - t0)
        return np.median(times) * 1000

    scale = D_bench ** -0.5
    t_std  = bench(lambda: standard_attention(Q_b, K_b, V_b, scale))
    t_sdpa = bench(lambda: F.scaled_dot_product_attention(Q_b, K_b, V_b, scale=scale))

    speedup = t_std / t_sdpa
    print(f"  {S_val:4d}   {t_std:10.3f}     {t_sdpa:12.3f}      {speedup:6.1f}×")

print()
print("Note: on CPU, speedup is modest (FlashAttention's main benefit is HBM BW reduction).")
print("On A100 GPU at S=2048: expect 3–5× speedup from HBM traffic reduction.")


#### What just happened — and what's missing

We measured the timing gap between standard attention and `F.scaled_dot_product_attention`. On CPU the speedup is modest — memory bandwidth isn't the primary bottleneck there — but the O(S²) vs. O(S) IO complexity means the gap *grows* with sequence length, confirming the theoretical prediction.

**What's missing:** We know FlashAttention is faster and we know *why* (HBM traffic). But `F.scaled_dot_product_attention` doesn't *always* dispatch to the FlashAttention kernel — it depends on dtype, shape, and hardware. Under what conditions does PyTorch actually use it? That's Part 5.

### Why fewer bytes can beat fewer FLOPsUse the lower bound $T\ge\max(\text{FLOPs}/\text{compute rate},\ \text{bytes}/\text{bandwidth})$. The larger term sets the pace.For **B=1, H=32, S=2048, d=128, bf16**, one S×S tensor contains $32\times2048^2=134{,}217{,}728$ values, about **268 MB**. Writing and reading scores, then writing and reading probabilities, moves about **1.07 GB**. At 2 TB/s that alone costs at least **0.54 ms**. Meanwhile QKᵀ plus PV is about **34.4 GFLOPs**, only **0.11 ms** at an ideal 312 TFLOP/s. Memory movement can therefore impose roughly five times the arithmetic lower bound.FlashAttention may do **more FLOPs** in backward by recomputing tiles yet finish sooner because it avoids moving stored S×S intermediates.**Misconceptions to retire**- **“FlashAttention approximates attention.”** It changes evaluation order and storage, not the function.- **“Fewer FLOPs always means faster.”** Memory-bound work waits on bytes.- **“The whole S×S matrix lives in SRAM.”** Only temporary tiles and running state do.- **“Fusion alone solves it.”** Tiling and online softmax make useful fusion fit in limited SRAM.- **“FlashAttention removes quadratic compute.”** Dense attention remains O(S²d) arithmetic. It removes quadratic materialization and reduces IO; the formal tiled IO bound depends on SRAM capacity M and is commonly written O(S²d²/M), not literally O(S).

---

## Part 5 — `scaled_dot_product_attention`: When Does PyTorch Use FlashAttention?

PyTorch 2.0's `F.scaled_dot_product_attention` automatically dispatches to the most efficient kernel available. But it only uses FlashAttention under specific conditions.

#### #### Predict first

At causal mask + **fp32** + S=512: does PyTorch 2.0 dispatch to FlashAttention?

1. **(a) Yes** — PyTorch always uses FlashAttention when available
2. **(b) No** — FlashAttention requires fp16 or bf16; fp32 falls back to standard computation
3. **(c) Depends on GPU** — only A100 and newer support it

In [ ]:
#  Part 5: SDPA dispatch conditions
S_val = 256
Q_test = torch.randn(2, S_val, D).to(DEVICE)
K_test = torch.randn(2, S_val, D).to(DEVICE)
V_test = torch.randn(2, S_val, D).to(DEVICE)

print("Testing F.scaled_dot_product_attention dispatch conditions:")
print()

test_cases = [
    ("fp32, no mask",           Q_test.float(),   K_test.float(),   V_test.float(),   None),
    ("fp16, no mask",           Q_test.half(),    K_test.half(),    V_test.half(),    None),
    ("bf16, no mask",           Q_test.bfloat16(),K_test.bfloat16(),V_test.bfloat16(),None),
    ("fp16, causal mask",       Q_test.half(),    K_test.half(),    V_test.half(),    "causal"),
]

# Try each dtype/mask combination and check whether SDPA runs and whether flash is dispatchable
for name, Q_c, K_c, V_c, mask in test_cases:
    try:
        with torch.backends.cuda.sdp_kernel(enable_flash=True, enable_math=True, enable_mem_efficient=True):
            if mask == "causal":
                out = F.scaled_dot_product_attention(Q_c, K_c, V_c, is_causal=True)
            else:
                out = F.scaled_dot_product_attention(Q_c, K_c, V_c)
        status = " ran"
    except Exception as e:
        status = f" error: {e}"

    # Check if flash was used by trying to force it
    flash_used = "unknown"
    if HAS_GPU:
        try:
            with torch.backends.cuda.sdp_kernel(enable_flash=True, enable_math=False, enable_mem_efficient=False):
                if mask == "causal":
                    _ = F.scaled_dot_product_attention(Q_c, K_c, V_c, is_causal=True)
                else:
                    _ = F.scaled_dot_product_attention(Q_c, K_c, V_c)
            flash_used = "flash available"
        except Exception:
            flash_used = "flash unavailable"
    else:
        flash_used = "CPU: no flash"

    print(f"  {name:30s}: {status:8s}  → {flash_used}")

print()
print("Prediction check: answer (b) — fp32 does not use FlashAttention on most GPUs.")
print("  FlashAttention requires dtype ∈ {fp16, bf16} on current hardware.")
print("  This is why training in bf16 (Ch2) unlocks faster attention — not just less memory!")


In [ ]:
# CPU context note
import torch

# Only show this note when running without a GPU
if not torch.cuda.is_available():
    print("\n CPU context: On CPU, all SDPA backends fall back to 'math' (standard attention).")
    print("   Flash Attention v2 requires CUDA — the efficiency gains are GPU-only.")
    print("   On an A100/H100, this cell would show 'flash' for sequences ≤ 65536 tokens.")
    print("   The algorithm you implemented in Parts 1-2 IS the same algorithm FlashAttention uses on GPU.")


#### What just happened — and what's missing

Prediction answer: **(b)** — fp32 does *not* dispatch to FlashAttention on current GPUs. The FlashAttention-2 CUDA kernel is implemented for fp16 and bf16 only. This is why the Ch2 recommendation to train in bf16 is load-bearing: it isn't just about memory savings — it's the gate that unlocks the 3–5× attention speedup.

**What's missing:** We've fixed attention compute (FlashAttention, Part 2–4) and confirmed the dispatch condition (Part 5). But there's a second memory bottleneck at *inference* time that FlashAttention doesn't address: the KV cache grows linearly with sequence length and number of heads. A 7B-model with full MHA needs ~0.5 GB of KV cache per concurrent request at S=2048. GQA and MQA in Part 6 attack that problem.

---

## Part 6 — GQA and MQA: When the KV Cache Becomes the Bottleneck

FlashAttention solved the *training* bottleneck: the S×S attention matrix no longer saturates HBM. But inference has a different memory problem entirely.

During autoregressive generation, each new token must attend over *all* previous tokens. To avoid recomputing K and V projections from scratch at every step, we cache them — the **KV cache**. For a 7B model with full MHA (32 heads, d_head=128), generating a 2048-token response requires holding 32 × 2048 key vectors and 32 × 2048 value vectors in GPU memory — *per layer, per request*. At 32 layers in bf16, that's **~0.5 GB per concurrent request, just for the KV cache**.

At 200 concurrent requests, you need 100 GB for KV caches alone — already beyond an A100's 80 GB of HBM, and we haven't counted model weights yet. The arithmetic is brutal.

**Grouped Query Attention (GQA)** and **Multi-Query Attention (MQA)** solve this by sharing K and V projections across multiple query heads. The key insight: Q can remain per-head (preserving attention's ability to specialize) while K and V are shared across groups of heads (slashing KV cache proportionally). LLaMA-3 and Mistral both use GQA-8 — 8× smaller KV cache, negligible quality regression.

#### #### Predict first

For a LLaMA-3-7B–style model (32 layers, d_head=128, bf16) generating a 2048-token sequence, MHA keeps KV cache for all 32 heads. GQA-8 uses 4 KV heads (8 query heads sharing each KV head). What is the GQA-8 KV cache size?

1. **(a) ~0.06 GB** — 8 query heads per KV head × 4 KV heads = 32× fewer KV vectors than MHA
2. **(b) ~0.2 GB** — GQA-8 means 8× fewer KV heads (32→4), so 8× reduction from MHA's ~1.6 GB
3. **(c) ~0.8 GB** — head sharing only applies to activations, not stored K/V cache

*Calculate by hand: 2 (K+V) × n_kv_heads × S × d_head × n_layers × bytes_per_element*

In [ ]:
#  Part 6: GQA and MQA — reducing KV cache size
print("Grouped Query Attention (GQA) and Multi-Query Attention (MQA):")
print()
print("Standard MHA: every head has its own Q, K, V projections")
print("GQA-8:        K and V shared across 8 heads; Q is per-head")
print("MQA:          K and V shared across ALL heads; Q is per-head")
print()

def kv_cache_size_gb(n_heads, n_kv_heads, seq_len, d_head, n_layers, dtype_bytes=2):
    """KV cache size in GB."""
    # 2 = K and V
    return 2 * n_kv_heads * seq_len * d_head * n_layers * dtype_bytes / 1e9

# LLaMA-3-7B config
N_LAYERS_LLM = 32
D_HEAD_LLM   = 128
SEQ          = 2048

configs = [
    ("MHA (all heads)",  32, 32),  # n_heads=32, n_kv_heads=32
    ("GQA-8",            32,  4),  # n_kv_heads = n_heads/8
    ("MQA",              32,  1),  # n_kv_heads = 1
]

print(f"KV cache at S={SEQ} for LLaMA-3-7B-style model ({N_LAYERS_LLM} layers, d_head={D_HEAD_LLM}):")
print(f"{'Config':20s}  {'n_kv_heads':10s}  {'KV cache (GB)':14s}  {'Reduction':10s}")
print("-" * 60)
kv_mha = kv_cache_size_gb(32, 32, SEQ, D_HEAD_LLM, N_LAYERS_LLM)

# Compare KV cache size for MHA, GQA-8, and MQA against the same MHA baseline
for name, n_heads, n_kv in configs:
    kv_gb = kv_cache_size_gb(n_heads, n_kv, SEQ, D_HEAD_LLM, N_LAYERS_LLM)
    reduction = f"{kv_mha/kv_gb:.0f}×" if kv_gb > 0 else "—"
    print(f"  {name:18s}  {n_kv:10d}  {kv_gb:12.2f} GB  {reduction:>8s}")

print()
print("Key insight: GQA-8 (used by LLaMA-3, Mistral) reduces KV cache by 8×")
print("with negligible quality loss vs. MHA — quality preserved because Q is still per-head.")
print()
print("Compute savings: standard attention takes O(S² × n_heads) time,")
print("GQA takes O(S² × n_kv_heads) time → 8× faster for equal quality.")


### Anchor 3 — Separate temporary attention IO from persistent KV-cache memoryFlashAttention asks **how can attention run without storing S×S intermediates?** GQA asks **how many distinct K/V sequences must decoding retain?** They attack different traffic and can be used together.![KV cache at S=2048: MHA (coral, largest) vs GQA-8 (teal, 8× smaller) vs MQA (amber, smallest)](images/kv-cache-memory-gqa.png)**Guided reading:** query heads stay at 32. Only cached K/V heads change, so KV-cache bytes scale directly with `n_kv_heads`.```mermaidflowchart LR M[MHA: 32 Q and 32 KV heads] -->|8 Q share each KV| G[GQA-8: 32 Q and 4 KV heads] G -->|all Q share one KV| Q[MQA: 32 Q and 1 KV head] M --> C1[Cache: 1×] G --> C2[Cache: 1/8] Q --> C3[Cache: 1/32] F[FlashAttention] --> R[Less temporary attention IO] G --> K[Less persistent decode cache] R --> Z[Compose both] K --> Z```GQA makes the retained cache and its read bandwidth proportionally smaller; it does not replace FlashAttention.

#### What just happened — and what's missing

Prediction answer: **(b)** — GQA-8 gives ~0.2 GB per request at S=2048. At 200 concurrent requests, that's 40 GB of KV cache (vs. the 320 GB that full MHA would require — 4× more than an A100 can hold).

The quality preservation comes from keeping Q per-head: each query can still express a unique attention pattern. Only the keys and values it attends *to* are shared across the group of 8 query heads. Empirically, GQA-8 matches full MHA across standard LLM benchmarks.

**What's missing:** We now have the complete picture — attention compute fixed (Parts 2–4), dispatch condition confirmed (Part 5), KV cache solved (Part 6). The Toy → Real bridge below maps every piece of this notebook's Python implementation to the production CUDA kernel.

####  Your Turn — GQA Group Size vs. KV Cache

GQA lets you trade KV cache memory for attention quality. The group size (query heads per KV head) controls the tradeoff. Try different `N_KV_HEADS_TRY` values to see how the memory scales — and where the LLaMA-3 / Mistral sweet spot of GQA-8 sits.

In [ ]:
#   Your Turn: GQA group size vs. KV cache memory
# # CHANGE: try different KV head counts to explore the quality-vs-memory tradeoff
N_KV_HEADS_TRY = 4    # ← CHANGE ME: try 1 (MQA), 2, 4, 8, 16, 32 (full MHA)
SEQ_TRY        = 4096 # ← CHANGE ME: try 1024, 8192

N_HEADS_TRY  = 32   # total query heads (fixed for this model config)
N_LAYERS_TRY = 32
D_HEAD_TRY   = 128
DTYPE_B_TRY  = 2    # bf16 bytes per element

# Compare this KV-head count's cache size against full MHA at the same sequence length
kv_gb_try     = 2 * N_KV_HEADS_TRY * SEQ_TRY * D_HEAD_TRY * N_LAYERS_TRY * DTYPE_B_TRY / 1e9
kv_mha_gb_try = 2 * N_HEADS_TRY * SEQ_TRY * D_HEAD_TRY * N_LAYERS_TRY * DTYPE_B_TRY / 1e9
reduction_try = kv_mha_gb_try / kv_gb_try if kv_gb_try > 0 else 0
group_size    = N_HEADS_TRY // N_KV_HEADS_TRY

print(f"Config: {N_HEADS_TRY} query heads / {N_KV_HEADS_TRY} KV heads  (group size = {group_size}×)")
print(f"Sequence length:    {SEQ_TRY} tokens")
print(f"KV cache — MHA:     {kv_mha_gb_try:.2f} GB")
print(f"KV cache — GQA-{group_size}:  {kv_gb_try:.2f} GB  ({reduction_try:.0f}× reduction)")
print()

# Name which attention variant this configuration corresponds to
if N_KV_HEADS_TRY == N_HEADS_TRY:
    print("→ Full MHA — every head has its own K and V. Maximum quality; maximum KV cache.")
elif N_KV_HEADS_TRY == 1:
    print("→ MQA — single K and V shared across all query heads. Smallest cache; some quality loss.")
elif group_size == 8:
    print("→ GQA-8 — matches LLaMA-3 / Mistral. 8× KV savings with negligible benchmark regression.")
else:
    print(f"→ GQA-{group_size} — {reduction_try:.0f}× savings. Quality risk grows as group size increases beyond ~8.")


---

##  Your Turn — Sequence Length Scaling

`SDPA`'s speedup over standard attention should grow as S increases (because the S×S IO grows faster than the O(S) tiled compute).

**Prediction:** When you double S from 512 to 1024, will the SDPA speedup ratio:
1. Stay roughly the same (~2-3×)
2. Increase (tiling advantage grows with S)
3. Decrease (overhead of tiling becomes dominant at larger S)

Run the cell below to find out.

In [ ]:
#   Your Turn: Speedup vs. sequence length
# # CHANGE: try different sequence lengths to see how the speedup ratio evolves
seq_lengths_exercise = [64, 128, 256, 512]  # ← CHANGE ME

print("SDPA vs. standard attention speedup at different sequence lengths:")
print(f"{'S':6s}  {'Standard (ms)':14s}  {'SDPA (ms)':12s}  {'Speedup':8s}  {'Prediction?'}")
print("-" * 65)

speedups = []

# Measure the SDPA speedup at each sequence length and note whether it's trending up
for S_val in seq_lengths_exercise:
    Q_e = torch.randn(B, S_val, D).to(DEVICE)
    K_e = torch.randn(B, S_val, D).to(DEVICE)
    V_e = torch.randn(B, S_val, D).to(DEVICE)
    scale = D ** -0.5

    # Time fn() n times with GPU sync around each call, return median ms
    def bench_simple(fn, n=20):
        if HAS_GPU: torch.cuda.synchronize()
        times = []
        for _ in range(n):
            if HAS_GPU: torch.cuda.synchronize()
            t0 = time.perf_counter()
            fn()
            if HAS_GPU: torch.cuda.synchronize()
            times.append(time.perf_counter() - t0)
        return np.median(times) * 1000

    t_std_e  = bench_simple(lambda: standard_attention(Q_e, K_e, V_e, scale))
    t_sdpa_e = bench_simple(lambda: F.scaled_dot_product_attention(Q_e, K_e, V_e, scale=scale))
    sp = t_std_e / t_sdpa_e
    speedups.append(sp)
    trend = "↑ growing" if len(speedups) > 1 and sp > speedups[-2] else ("→ stable" if len(speedups) > 1 else "  first")
    print(f"  {S_val:4d}   {t_std_e:10.3f}     {t_sdpa_e:10.3f}    {sp:6.1f}×  {trend}")

print()

# Confirm whether the overall trend across all tested lengths was growing or flat
if len(speedups) > 1 and speedups[-1] > speedups[0]:
    print("→ Speedup grows with S — confirms tiling advantage increases quadratically with sequence length")
else:
    print("→ Speedup is approximately constant — on this hardware/problem size the bottleneck may be compute")


---

## Toy → Production Bridge

This notebook built a readable Python implementation of FlashAttention. Here is how every piece maps to the production CUDA kernel:

| Concept | Toy (this notebook) | Production (FlashAttention-2 / PyTorch 2.0) |
|---|---|---|
| Score matrix | `(B, S, S)` float32 tensor in Python | Never materialized; exists only in GPU registers inside the Triton/CUDA kernel |
| "SRAM" | Python variables `M_i, L_i, O_i` | On-chip shared memory per streaming multiprocessor (228 KB/SM on A100) |
| Block size | `BLOCK_SIZE=32` (arbitrary integer) | Tuned per GPU: 64–128, selected at compile time for register pressure and occupancy |
| Online softmax | `exp(M_i - M_new)` correction in fp32 | Same math; accumulator kept in fp32 while K/V/Q inputs are fp16/bf16 |
| HBM writes | Python tensor slice assignment | One coalesced write per output tile, issued once per Q-block |
| Backward pass | Not shown (autograd handles it) | Recomputes S_ij tiles from stored Q, K, V during backward — trades HBM reads for FLOPs |
| Causal mask | Not shown | Skips lower-triangular tiles entirely; saves ~2× work for causal LM |
| API surface | `tiled_attention_forward(Q, K, V)` | `F.scaled_dot_product_attention(Q, K, V, is_causal=True)` in bf16 |
| Observed speedup | ~1× (Python overhead dominates) | 3–5× over eager attention on A100 at S=2048 |

---## Summary| Part | Concept | Key result ||------|---------|----------|| 1 | Standard attention IO | O(S²) HBM traffic — score matrix read 5–6× during fwd+bwd || 2 | Tiling | Full forward pass without writing S×S to HBM — proved with Python impl; `torch.allclose` exact || 3 | Online softmax | Algebraically exact and verified with `torch.allclose`; numerically stable under extreme inputs || 4 | IO complexity | Standard O(S²), FlashAttention O(S) — 3–5× speedup on GPU; grows with sequence length || 5 | SDPA dispatch | fp32 uses standard path; fp16/bf16 triggers FlashAttention — dtype is the gate || 6 | GQA/MQA | 8× KV cache reduction (LLaMA-3/Mistral config) with negligible quality loss |---### Key insights to keep- **The bottleneck was the bus, not the arithmetic.** Standard attention is slow because the S×S matrix makes 5–6 HBM roundtrips per forward-backward pass — not because softmax is computationally expensive.- **Tiling is exact, not approximate.** FlashAttention preserves standard attention; implementations agree within floating-point tolerance. The online softmax correction is algebraically exact — no precision is traded away.- **fp16/bf16 is the FlashAttention dispatch gate.** `F.scaled_dot_product_attention` only uses the FlashAttention kernel for half-precision inputs. This is a second reason bf16 training is the 2024 default — it unlocks the attention speedup, not just memory savings.- **GQA-8 halves the inference memory crisis.** Sharing K and V across 8 query heads (LLaMA-3, Mistral) reduces KV cache 8× with negligible benchmark regression. Q remains per-head — quality is preserved where it matters.- **One-line fix, 60%+ latency recovered.** The opening profiling scenario resolves by switching to bf16 + calling `F.scaled_dot_product_attention`. The S×S matrix disappears from HBM entirely.

In [ ]:
#  Closing Decision
# Summarize the IO savings at S=512 (the Ch3 bottleneck case)
S_close = 512
std_hbm = measure_hbm_traffic_gb(B, S_close, D) * 6  # forward + backward ≈ 6× forward
flash_hbm = measure_hbm_traffic_gb(B, S_close, D)    # FlashAttention ≈ 1× forward (O(S))
ratio = std_hbm / flash_hbm

# Describe which dispatch path SDPA will actually take on this machine
dispatch_str = "fp16/bf16 required (fp32 uses standard path)" if not HAS_GPU else (
    f"{torch.cuda.get_device_properties(0).name} — fp16/bf16 triggers flash kernel")

print("=" * 60)
print("  CLOSING DECISION — FlashAttention for S=512 Bottleneck")
print("=" * 60)
print()
print(f"  Standard attention (fwd+bwd): ~{std_hbm:.2f} GB HBM reads")
print(f"  FlashAttention:               ~{flash_hbm:.2f} GB HBM reads")
print(f"  IO reduction:                  {ratio:.1f}× less HBM traffic")
print()
print(f"  Dispatch condition: {dispatch_str}")
print()
print("  RECOMMENDATION:")
print("  1. Use F.scaled_dot_product_attention() — zero code change, automatic dispatch")
print("  2. Train in bf16 (Ch2) to ensure FlashAttention is triggered")
print("  3. For causal LM: pass is_causal=True (enables more efficient masking)")
print("  4. For GQA models (LLaMA-3, Mistral): n_kv_heads = n_heads/8 → 8× KV cache savings")
print()
print("  One-line change that captures 60%+ of the attention bottleneck:")
print("  # Before:")
print("  #   attn = softmax(Q @ K.T / sqrt(d)) @ V")
print("  # After:")
print("  #   attn = F.scaled_dot_product_attention(Q, K, V)  ← FlashAttention dispatches here")


---

## What This Notebook Covered (and What It Didn't)

### Tier 1 — Implemented and Demonstrated
- Standard attention IO — measured HBM traffic growth as O(S²)
- Tiling algorithm — clean Python implementation; passes `torch.allclose` vs. reference
- Online softmax — proven bit-identical to `torch.softmax`; numerical stability demonstrated
- IO complexity — timing measured; relative speedup shown across sequence lengths
- SDPA dispatch — tested fp32 vs fp16/bf16; dispatch condition confirmed
- GQA/MQA — KV cache size computed for LLaMA-3-7B config

### Tier 2 — Explained, Not Fully Built
- **GQA at toy scale** — the KV sharing mechanism is explained; a toy implementation exists in the spec but was replaced with the memory sizing analysis (more informative for the deployment question)

### Tier 3 — Named but Out of Scope
- **FlashAttention-3** — uses asynchronous data movement with the new H100 Tensor Memory Accelerator; 2× faster than FA-2 on H100
- **Ring Attention** — extends FlashAttention to multi-GPU sequence parallelism; each GPU handles one "ring" of the sequence
- **Sliding Window Attention** — limits attention span to a local window; used in Mistral for very long contexts

---

## When to Use What

| Situation | Action | Reason |
|---|---|---|
| Any transformer inference | Use `F.scaled_dot_product_attention` | Auto-dispatches to FlashAttention if conditions met |
| Training with long sequences | Use bf16 precision | Enables FlashAttention dispatch |
| Deploying 7B+ models | Use GQA (LLaMA-3/Mistral) | 8× KV cache savings = 8× more context per GPU |
| Custom attention variant not covered | Write Triton kernel (Ch8) | Production FlashAttention-2 repo as reference |

→ **Next:** `learning/ai-infrastructure/07-inference-systems/` — FlashAttention reduces per-token latency. The next chapter covers the system-level optimizations: KV cache, continuous batching, speculative decoding.

---

## Production and Cloud Deployment: Flash / SDPA Attention

Treat attention backend selection as a release policy, not a one-time hardware assumption. Pin and record the PyTorch build, CUDA runtime, driver/container image, GPU model and compute capability; re-run qualification whenever any of them changes. PyTorch's SDPA dispatcher evolves across releases, so a dtype or shape that uses FlashAttention today may select an efficient, cuDNN, or math implementation after an upgrade.

- **Capability and version gates:** require a supported PyTorch SDPA API, CUDA visibility for fused kernels, and an approved device/runtime combination. A forced-backend smoke call is the final capability check because compile-time availability alone does not prove that a particular input is dispatchable.
- **Backend policy:** prefer Flash SDPA for qualified CUDA `float16`/`bfloat16` workloads, allow the dispatcher to choose among approved fused kernels when explicit Flash is unavailable, and retain the math backend as the deterministic rollback path. Never infer the backend from latency alone.
- **Shape and dtype constraints:** validate Q/K/V rank, batch/head geometry, sequence dimensions, head dimension, contiguity, device, dtype, mask semantics, dropout, and GQA settings against representative production requests. Kernel constraints vary by PyTorch and GPU generation; test the exact serving shapes rather than encoding a permanent head-dimension rule.
- **Correctness parity:** compare candidate output with forced math SDPA using fixed inputs and dtype-appropriate tolerances. Include causal/non-causal masks, padding, long-context boundaries, and backward gradients for training releases. Non-finite output or a tolerance breach blocks promotion.
- **Benchmark gates:** warm up first, synchronize CUDA around timing, and gate p50/p95 latency plus memory on representative batch, head, sequence, and head-dimension buckets. A tiny smoke benchmark catches gross regressions but does not replace load tests with production concurrency.
- **Deterministic fallback:** route failed qualification to forced math SDPA, disable dropout for inference, and use your platform's deterministic-algorithm policy where required. Determinism can reduce performance and remains version/platform dependent, so qualify it separately.
- **Telemetry and rollback:** emit policy/version, requested and selected backend, shape bucket, dtype, device, latency, parity error, fallback reason, and failure counts without tensor contents. Canary a new image or policy, retain the previous report and deployment, and roll back automatically when correctness, error-rate, or benchmark gates fail.

The cells below use this notebook's `torch`, `F.scaled_dot_product_attention`, `DEVICE`, and `torch.backends.cuda.sdp_kernel` APIs. They are cloud-neutral: a deployment controller can consume the persisted JSON report as promotion evidence, while the serving layer exports the same fields to its metrics/tracing backend.

In [ ]:
from contextlib import nullcontext
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from time import perf_counter
from typing import Any, Mapping
import inspect
import json
import statistics


@dataclass(frozen=True)
class AttentionDeploymentPolicy:
    policy_version: str = "sdpa-production-v1"
    minimum_torch_version: tuple[int, int] = (2, 0)
    preferred_backend: str = "flash"
    fallback_backend: str = "math"
    parity_atol: float = 1e-2
    parity_rtol: float = 1e-2
    max_candidate_to_math_latency_ratio: float = 1.05
    warmup_iterations: int = 3
    benchmark_iterations: int = 10
    report_path: Path = Path("artifacts/flash-attention/sdpa-qualification.json")


RUN_PRODUCTION_PARITY = False
RUN_PRODUCTION_BENCHMARK = False
RUN_PRODUCTION_REPORT = False
PRODUCTION_ATTENTION_POLICY = AttentionDeploymentPolicy()


def _torch_major_minor() -> tuple[int, int]:
    numeric = torch.__version__.split("+", maxsplit=1)[0].split(".")
    return int(numeric[0]), int(numeric[1])


def _validate_sdpa_inputs(query: torch.Tensor, key: torch.Tensor, value: torch.Tensor) -> None:
    if query.ndim not in (3, 4) or key.ndim != query.ndim or value.ndim != query.ndim:
        raise ValueError("Q, K, and V must all be rank 3 or rank 4 tensors.")
    if query.device != key.device or query.device != value.device:
        raise ValueError("Q, K, and V must be on the same device.")
    if query.dtype != key.dtype or query.dtype != value.dtype:
        raise ValueError("Q, K, and V must use the same dtype.")
    if query.shape[:-2] != key.shape[:-2] or key.shape[:-2] != value.shape[:-2]:
        raise ValueError("This policy expects matching batch/head dimensions; qualify GQA separately.")
    if query.shape[-1] != key.shape[-1] or key.shape[-2] != value.shape[-2]:
        raise ValueError("Q/K head dimensions and K/V sequence lengths must match.")


def _forced_sdpa_backend(backend: str):
    if backend == "auto":
        return nullcontext()
    if backend not in {"flash", "efficient", "math"}:
        raise ValueError(f"Unsupported SDPA backend policy: {backend}")

    kwargs = {
        "enable_flash": backend == "flash",
        "enable_math": backend == "math",
        "enable_mem_efficient": backend == "efficient",
    }
    try:
        parameters = inspect.signature(torch.backends.cuda.sdp_kernel).parameters
    except (TypeError, ValueError):
        parameters = {}
    if "enable_cudnn" in parameters:
        kwargs["enable_cudnn"] = False
    return torch.backends.cuda.sdp_kernel(**kwargs)


def select_sdpa_policy(
    query: torch.Tensor,
    key: torch.Tensor,
    value: torch.Tensor,
    policy: AttentionDeploymentPolicy = PRODUCTION_ATTENTION_POLICY,
    require_deterministic: bool = False,
) -> dict[str, Any]:
    """Select a candidate backend from observable gates; forced smoke tests make the final decision."""
    _validate_sdpa_inputs(query, key, value)
    if _torch_major_minor() < policy.minimum_torch_version:
        raise RuntimeError(
            f"PyTorch {torch.__version__} is below the SDPA policy minimum "
            f"{policy.minimum_torch_version}."
        )

    reasons: list[str] = []
    flash_build_available = bool(
        getattr(torch.backends.cuda, "is_flash_attention_available", lambda: False)()
    )
    if require_deterministic:
        selected = policy.fallback_backend
        reasons.append("deterministic policy requested")
    elif query.device.type != "cuda":
        selected = policy.fallback_backend
        reasons.append("fused CUDA SDPA requires CUDA tensors")
    elif query.dtype not in {torch.float16, torch.bfloat16}:
        selected = policy.fallback_backend
        reasons.append("Flash SDPA qualification requires float16 or bfloat16 inputs")
    elif policy.preferred_backend == "flash" and flash_build_available:
        selected = "flash"
        reasons.append("CUDA half-precision input and Flash SDPA build capability detected")
    else:
        selected = "auto"
        reasons.append("dispatcher must select an available fused or fallback backend")

    device_name = (
        torch.cuda.get_device_name(query.device) if query.device.type == "cuda" else str(query.device)
    )
    capability = (
        list(torch.cuda.get_device_capability(query.device))
        if query.device.type == "cuda"
        else None
    )
    return {
        "policy_version": policy.policy_version,
        "torch_version": torch.__version__,
        "cuda_runtime": torch.version.cuda,
        "device": device_name,
        "compute_capability": capability,
        "shape": list(query.shape),
        "dtype": str(query.dtype),
        "contiguous": all(tensor.is_contiguous() for tensor in (query, key, value)),
        "flash_build_available": flash_build_available,
        "selected_backend": selected,
        "fallback_backend": policy.fallback_backend,
        "selection_reasons": reasons,
    }

In [ ]:
def _run_sdpa(
    query: torch.Tensor,
    key: torch.Tensor,
    value: torch.Tensor,
    backend: str,
    is_causal: bool,
) -> torch.Tensor:
    with torch.inference_mode(), _forced_sdpa_backend(backend):
        return F.scaled_dot_product_attention(
            query, key, value, dropout_p=0.0, is_causal=is_causal
        )


def run_sdpa_parity_smoke(
    query: torch.Tensor,
    key: torch.Tensor,
    value: torch.Tensor,
    policy: AttentionDeploymentPolicy = PRODUCTION_ATTENTION_POLICY,
    is_causal: bool = True,
) -> dict[str, Any]:
    """Compare the selected backend with forced math SDPA on one bounded input."""
    selection = select_sdpa_policy(query, key, value, policy)
    candidate_backend = str(selection["selected_backend"])
    report: dict[str, Any] = {**selection, "is_causal": is_causal}

    reference = _run_sdpa(query, key, value, policy.fallback_backend, is_causal)
    try:
        candidate = _run_sdpa(query, key, value, candidate_backend, is_causal)
        finite = bool(torch.isfinite(candidate).all().item())
        max_abs_error = float((candidate.float() - reference.float()).abs().max().item())
        parity_passed = finite and torch.allclose(
            candidate.float(),
            reference.float(),
            atol=policy.parity_atol,
            rtol=policy.parity_rtol,
        )
        report.update(
            candidate_available=True,
            finite_output=finite,
            max_abs_error=max_abs_error,
            parity_passed=bool(parity_passed),
        )
    except Exception as error:
        report.update(
            candidate_available=False,
            finite_output=False,
            max_abs_error=None,
            parity_passed=False,
            candidate_error=f"{type(error).__name__}: {error}",
        )

    if not report["candidate_available"] or not report["parity_passed"]:
        report["decision"] = "ROLL_BACK_TO_MATH"
        report["active_backend"] = policy.fallback_backend
    elif candidate_backend == policy.fallback_backend:
        report["decision"] = "USE_MATH_FALLBACK"
        report["active_backend"] = policy.fallback_backend
    else:
        report["decision"] = "CANDIDATE_PARITY_PASSED"
        report["active_backend"] = candidate_backend
    return report


def _benchmark_sdpa_backend(
    query: torch.Tensor,
    key: torch.Tensor,
    value: torch.Tensor,
    backend: str,
    is_causal: bool,
    warmup_iterations: int,
    benchmark_iterations: int,
) -> dict[str, float]:
    for _ in range(warmup_iterations):
        _run_sdpa(query, key, value, backend, is_causal)
    if query.device.type == "cuda":
        torch.cuda.synchronize(query.device)

    samples_ms = []
    for _ in range(benchmark_iterations):
        started = perf_counter()
        _run_sdpa(query, key, value, backend, is_causal)
        if query.device.type == "cuda":
            torch.cuda.synchronize(query.device)
        samples_ms.append((perf_counter() - started) * 1000)

    ordered = sorted(samples_ms)
    p95_index = min(len(ordered) - 1, max(0, int(0.95 * len(ordered))))
    return {
        "p50_ms": float(statistics.median(samples_ms)),
        "p95_ms": float(ordered[p95_index]),
        "iterations": float(benchmark_iterations),
    }


def run_sdpa_benchmark_smoke(
    query: torch.Tensor,
    key: torch.Tensor,
    value: torch.Tensor,
    candidate_backend: str,
    policy: AttentionDeploymentPolicy = PRODUCTION_ATTENTION_POLICY,
    is_causal: bool = True,
) -> dict[str, Any]:
    """Run a bounded synchronized smoke benchmark; use representative load tests for release gates."""
    candidate = _benchmark_sdpa_backend(
        query,
        key,
        value,
        candidate_backend,
        is_causal,
        policy.warmup_iterations,
        policy.benchmark_iterations,
    )
    math = _benchmark_sdpa_backend(
        query,
        key,
        value,
        policy.fallback_backend,
        is_causal,
        policy.warmup_iterations,
        policy.benchmark_iterations,
    )
    ratio = candidate["p50_ms"] / math["p50_ms"]
    return {
        "candidate_backend": candidate_backend,
        "candidate": candidate,
        "math": math,
        "candidate_to_math_p50_ratio": ratio,
        "benchmark_passed": ratio <= policy.max_candidate_to_math_latency_ratio,
        "maximum_allowed_ratio": policy.max_candidate_to_math_latency_ratio,
    }


def persist_sdpa_qualification_report(
    report: Mapping[str, Any],
    path: Path = PRODUCTION_ATTENTION_POLICY.report_path,
) -> Path:
    """Atomically persist promotion evidence for CI/CD or a deployment controller."""
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(dict(report), indent=2, sort_keys=True, default=str),
        encoding="utf-8",
    )
    temporary.replace(path)
    return path

In [ ]:
PRODUCTION_SMOKE_SHAPE = (1, 4, 256, 64)
PRODUCTION_SMOKE_IS_CAUSAL = True


if RUN_PRODUCTION_PARITY or RUN_PRODUCTION_BENCHMARK or RUN_PRODUCTION_REPORT:
    smoke_dtype = (
        torch.bfloat16
        if DEVICE.type == "cuda" and torch.cuda.is_bf16_supported()
        else torch.float16
        if DEVICE.type == "cuda"
        else torch.float32
    )
    generator = torch.Generator(device=DEVICE).manual_seed(42)
    smoke_query = torch.randn(
        PRODUCTION_SMOKE_SHAPE, device=DEVICE, dtype=smoke_dtype, generator=generator
    ).contiguous()
    smoke_key = torch.randn(
        PRODUCTION_SMOKE_SHAPE, device=DEVICE, dtype=smoke_dtype, generator=generator
    ).contiguous()
    smoke_value = torch.randn(
        PRODUCTION_SMOKE_SHAPE, device=DEVICE, dtype=smoke_dtype, generator=generator
    ).contiguous()

    selection_report = select_sdpa_policy(
        smoke_query, smoke_key, smoke_value, PRODUCTION_ATTENTION_POLICY
    )
    parity_report: dict[str, Any] = {"status": "SKIPPED"}
    benchmark_report: dict[str, Any] = {"status": "SKIPPED"}

    if RUN_PRODUCTION_PARITY:
        parity_report = run_sdpa_parity_smoke(
            smoke_query,
            smoke_key,
            smoke_value,
            PRODUCTION_ATTENTION_POLICY,
            is_causal=PRODUCTION_SMOKE_IS_CAUSAL,
        )

    parity_allows_benchmark = (
        not RUN_PRODUCTION_PARITY or bool(parity_report.get("parity_passed"))
    )
    if RUN_PRODUCTION_BENCHMARK and parity_allows_benchmark:
        benchmark_report = run_sdpa_benchmark_smoke(
            smoke_query,
            smoke_key,
            smoke_value,
            candidate_backend=str(selection_report["selected_backend"]),
            policy=PRODUCTION_ATTENTION_POLICY,
            is_causal=PRODUCTION_SMOKE_IS_CAUSAL,
        )
    elif RUN_PRODUCTION_BENCHMARK:
        benchmark_report = {"status": "SKIPPED_AFTER_PARITY_FAILURE"}

    parity_passed = not RUN_PRODUCTION_PARITY or bool(parity_report.get("parity_passed"))
    benchmark_passed = not RUN_PRODUCTION_BENCHMARK or bool(
        benchmark_report.get("benchmark_passed")
    )
    qualification_passed = parity_passed and benchmark_passed
    selected_backend = str(selection_report["selected_backend"])
    if not qualification_passed:
        decision = "ROLL_BACK_TO_MATH"
        active_backend = PRODUCTION_ATTENTION_POLICY.fallback_backend
    elif selected_backend == PRODUCTION_ATTENTION_POLICY.fallback_backend:
        decision = "KEEP_MATH_FALLBACK"
        active_backend = PRODUCTION_ATTENTION_POLICY.fallback_backend
    else:
        decision = "PROMOTE_CANDIDATE"
        active_backend = selected_backend

    policy_record = asdict(PRODUCTION_ATTENTION_POLICY)
    policy_record["report_path"] = str(PRODUCTION_ATTENTION_POLICY.report_path)
    production_qualification_report = {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "policy": policy_record,
        "selection": selection_report,
        "parity": parity_report,
        "benchmark": benchmark_report,
        "decision": decision,
        "active_backend": active_backend,
        "telemetry_fields": [
            "policy_version",
            "torch_version",
            "cuda_runtime",
            "device",
            "compute_capability",
            "shape",
            "dtype",
            "selected_backend",
            "active_backend",
            "latency_ms",
            "max_abs_error",
            "fallback_reason",
            "error_count",
        ],
    }

    print(json.dumps(production_qualification_report, indent=2, default=str))
    if RUN_PRODUCTION_REPORT:
        persisted_report = persist_sdpa_qualification_report(
            production_qualification_report,
            PRODUCTION_ATTENTION_POLICY.report_path,
        )
        print(f"Persisted SDPA qualification report: {persisted_report}")
else:
    print(
        "Production SDPA checks are disabled; set RUN_PRODUCTION_PARITY, "
        "RUN_PRODUCTION_BENCHMARK, and/or RUN_PRODUCTION_REPORT explicitly to run."
    )